In [ ]:
import os
os.chdir('..')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from hdbscan import HDBSCAN
from umap import UMAP
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer
from yellowbrick.cluster import KElbowVisualizer

df = pd.read_csv("data/all_shorts_comments.csv", encoding='utf-8',lineterminator='\n')
comments = df['comment'].tolist()

In [ ]:
model = SentenceTransformer('jhgan/ko-sroberta-multitask')
embeddings = model.encode(comments, show_progress_bar=True)

In [ ]:
umap_model = UMAP(n_components=15, random_state=123)
reduced_embeddings = umap_model.fit_transform(embeddings)

In [ ]:
vzr = KElbowVisualizer(KMeans(random_state=123), k=(2, 20))
vzr.fit(reduced_embeddings)
vzr.poof()

In [ ]:
kmeans = KMeans(n_clusters=6, random_state=123)
kmeans_labels = kmeans.fit_predict(reduced_embeddings)

hdbscan_clusterer = HDBSCAN(min_cluster_size=15)
hdbscan_labels = hdbscan_clusterer.fit_predict(reduced_embeddings)

In [ ]:
pca = PCA(n_components=2)
reduced = pca.fit_transform(reduced_embeddings)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(reduced[:, 0], reduced[:, 1], c=kmeans_labels, cmap='tab10', s=30)
axes[0].set_title("KMeans Clustering (k=6)")

axes[1].scatter(reduced[:, 0], reduced[:, 1], c=hdbscan_labels, cmap='tab10', s=30)
axes[1].set_title("HDBSCAN Clustering")

plt.tight_layout()
plt.show()

In [ ]:
df["kmeans"] = kmeans.labels_
df["hdbscan"] = hdbscan_clusterer.labels_

In [ ]:
df.to_csv("data/cluster_shorts.csv", index=False)